## invoke

阻塞式调用,直到模型返回结果,参数有input,config,input可输入类型为list[str] | tuple[str, str] | str | dict[str, Any]]


In [1]:
import os
import dotenv
from aiohttp.hdrs import FROM
from langchain_openai import ChatOpenAI

dotenv.load_dotenv(override= True)
DEEPSEEK_API_KEY=os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL=os.getenv("DEEPSEEK_BASE_URL")

llm_deepseek = ChatOpenAI(
    api_key= DEEPSEEK_API_KEY,
    base_url= DEEPSEEK_BASE_URL,
    model= "deepseek-v4-flash"
)

In [2]:
from langchain_core.messages import HumanMessage
from langchain_core.messages import SystemMessage

message = [
    SystemMessage(content="你是一个专业的翻译"),
    HumanMessage(content="白雪皑皑"),
    ]

print(llm_deepseek.invoke(message))


content='An expanse of white snow / Snow-covered landscape / Pure white snow' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 179, 'prompt_tokens': 92, 'total_tokens': 271, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 165, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 92}, 'model_provider': 'openai', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_a18b46594c_prod0820_fp8_kvcache_20260402', 'id': '0062798b-2fa6-463d-ba14-827adc8b6a56', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019fc180-87ec-7781-8e4b-efde94107cfe-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 92, 'output_tokens': 179, 'total_tokens': 271, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 165}}


In [3]:
from rich import print as r_print
#可以使用rich打印,会更美观
r_print(llm_deepseek.invoke(message))


AIMessage(
    content='“白雪皑皑” 的英文翻译：\n\n**"White snow covering the ground"** 或更诗意的表达：**"Pure white 
snow"**\n\n如果想更文学化一些，可以用：\n- *The ground is covered with gleaming white snow.*\n- *A vast expanse of 
white snow.*\n\n“皑皑”形容雪洁白明亮的样子，所以翻译时要体现出“洁白”、“覆盖大地”的意象。',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 239,
            'prompt_tokens': 92,
            'total_tokens': 331,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 149,
                'rejected_prediction_tokens': None
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
            'prompt_cache_hit_tokens': 0,
            'prompt_cache_miss_tokens': 92
        },
        'model_provider': 'openai',
        'model_name': 'deepseek-v4-flash',
        'system_fingerprint': 'fp_a18b46594c_prod0820_fp8_kvcache_20260402',
        'id': '8bb0a1b6-f180-4bc8-abc8-1b87185ae2f6',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--019fc182-0a80-7c01-93a2-1d00013494c5-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 92,
        'output_tokens': 239,
        'total_tokens': 331,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 149}
    }
)

## stream
流式调用,返回一个可迭代对象,每次迭代返回一个token


In [5]:
for chunk in llm_deepseek.stream(message):
    print(chunk.text, end='', flush=True)


The snow is gleaming white.

## batch
允许你一次性发送一组请求（含多条独立请求）,模型会在后台 并行处理,然后返回所有结果的列表。
对于多个请求,模型会并行处理,并返回所有结果的列表,缩短了多个请求的网络请求时间。


In [6]:
messages = [
    [
        SystemMessage(content="你是一个专业的翻译"),
        HumanMessage(content="白雪皑皑"),
    ],
    [
        SystemMessage(content="你是一个数学家"),
        HumanMessage(content="简要说明泰勒展开"),
    ],
]

print(llm_deepseek.batch(messages)[0].content)
print(llm_deepseek.batch(messages)[1].content)


The snow is pure white.
泰勒展开是一种用多项式函数近似表示一般函数的方法。

对于一个在 \(x=a\) 处具有任意阶导数的函数 \(f(x)\)，它的泰勒展开式为：

\[
f(x)=\sum_{n=0}^{\infty}\frac{f^{(n)}(a)}{n!}(x-a)^n
\]

其中 \(f^{(n)}(a)\) 是 \(f(x)\) 在 \(x=a\) 处的 \(n\) 阶导数。

当 \(a=0\) 时，这个展开式也叫**麦克劳林展开**：

\[
f(x)=\sum_{n=0}^{\infty}\frac{f^{(n)}(0)}{n!}x^n
\]

实际应用中，我们通常只取有限项来近似，截断后的余项称为拉格朗日余项或佩亚诺余项。

**核心思想**：用幂函数的线性组合去逼近复杂的函数，展开点附近的近似效果最好，离展开点越远误差通常越大。

例如，\(e^x\) 在 \(x=0\) 处的泰勒展开为：

\[
e^x=1+x+\frac{x^2}{2!}+\frac{x^3}{3!}+\cdots
\]

取前几项就可以在 \(x=0\) 附近近似计算 \(e^x\)。


## ainvoke,astream,abatch异步调用
在主线程中调用,不阻塞主线程的同时,返回一个可等待对象,等待模型返回结果后,再返回结果
```python
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
import asyncio
import time
# 从.env文件中加载环境变量
load_dotenv(override=True)
CLOSEAI_API_KEY = os.getenv("CLOSEAI_API_KEY")
CLOSEAI_BASE_URL = os.getenv("CLOSEAI_BASE_URL")
model = init_chat_model(
    model="openai:gpt-5.4-mini",
    api_key=CLOSEAI_API_KEY,
    base_url=CLOSEAI_BASE_URL
    )
async def demo_async_invoke():
    print("=== 演示：ainvoke 的异步（非阻塞）效果 ===")
    start_time = time.perf_counter() # 记录开始时间
    print("程序开始...")
    # 1. 创建任务 (Task)
    print(">>> 发起异步模型调用 (ainvoke)...")
    async_task = asyncio.create_task(model.ainvoke("用一句话解释人工智能。"))
    # 2. 并行执行其他任务
    print(">>> 模型请求已在后台发送，继续执行本地逻辑...")
    for i in range(3):
        await asyncio.sleep(1) # 使用异步等待，释放控制权
    print(f">>> 正在执行第{i + 1}个任务... (已耗时 {time.perf_counter() - start_time:.2f}s)")
    # 3. 获取模型结果
    print(">>> 本地任务完成，检查模型状态...")
    response = await async_task
    end_time = time.perf_counter()
    print(f">>> 模型返回: {response.content}")
    print(f"=== 总运行耗时: {end_time - start_time:.2f}s ===")

async def main():
    """主函数"""
    await demo_async_invoke()

if __name__ == "__main__":
    asyncio.run(main())
```